Nesse código de prática irei realizar 6 trechos de código, objetivando pegar todos os passos feito no código da aula, o primeiro trecho vai conter as instalações e importações, o segundo as chaves API dos modelos usados, o terceiro é somente a criação da sessão, o quarto as funções usadas para cada um dos agentes, o quinto são as criações dos agentes, o sexto a função principal, e, por último, as chamadas da função principal, com as queries.

Instalações e importações das bibliotecas usadas.

In [1]:
%pip install google-adk -q
%pip install litellm -q
%pip install mistralai 

import os
import asyncio
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types
from google.adk.tools.tool_context import ToolContext
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.models.llm_response import LlmResponse
from google.adk.tools.base_tool import BaseTool
from typing import Optional, Dict, Any

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Definição das chaves API dos três modelos que iremos usar, no caso escolhi três modelos que estavam a minha disposição no momento, e, como vou usar o gemini, também desabilitei o vertex.

In [ ]:
os.environ["GOOGLE_API_KEY"] = "COLOQUE A CHAVE API AQUI"
os.environ['GROQ_API_KEY'] = 'COLOQUE A CHAVE API AQUI'
os.environ['MISTRAL_API_KEY'] = 'COLOQUE A CHAVE API AQUI'
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

Criação da sessão (memória).

In [3]:
session_service = InMemorySessionService()

session = await session_service.create_session(
    app_name="gerenciador_de_matricula",
    user_id="usuario_1",
    session_id="sessao_001"
)

O sistema será um gerenciador de matrículas em matérias, para isso temos que implementar 5 ferramentas: get_materias e get_materia para um agente, get_materias_matriculadas, matricular e desmatricular para outro agente.

In [4]:
MATERIAS_DB = [
{
    "id": 1,
    "nome": "Algoritmos",
    "carga_horaria": 80,
    "professor": "Dr. Carlos",
    "ementa": "Estruturas de dados básicas, análise de complexidade.",
    "ativo" : True
},
{
    "id": 2,
    "nome": "Banco de Dados",
    "carga_horaria": 60,
    "professor": "Profa. Mariana",
    "ementa": "Modelagem relacional, SQL, normalização.",
    "ativo" : True
},
{
    "id": 3,
    "nome": "Inteligência Artificial",
    "carga_horaria": 80,
    "professor": "Dr. Henrique",
    "ementa": "Busca, aprendizado de máquina, agentes inteligentes.",
    "ativo" : False
},
{
    "id": 4,
    "nome": "Programação Orientada a Objetos",
    "carga_horaria": 80,
    "professor": "Dr. Renato",
    "ementa": "Paradigma de programação, modularização, encapsulamento.",
    "ativo" : True
}
]

MATERIAS_MATRICULADAS_DB = []
def get_materias() -> list:
    return [
        {
            "id": materia["id"],
            "nome": materia["nome"],
            "carga_horaria": materia["carga_horaria"],
            "professor": materia["professor"]
        }
        for materia in MATERIAS_DB if materia["ativo"]
    ]
def get_materia(nome: str) -> dict:
    nome = nome.lower()

    for materia in MATERIAS_DB:
        if materia["nome"].lower() == nome:
            return materia

    return {"erro": "Matéria não encontrada."}

async def get_materias_matriculadas(tool_context: ToolContext):

    state = tool_context.state

    if "materias_matriculadas" not in state:
        state["materias_matriculadas"] = []

    materias_ids = state["materias_matriculadas"]

    if not materias_ids:
        return "Você ainda não está matriculado em nenhuma matéria."
    
    materias_completas = [
        {
            "id": materia["id"],
            "nome": materia["nome"],
            "carga_horaria": materia["carga_horaria"],
            "professor": materia["professor"]
        }
        for materia in MATERIAS_DB
        if materia["id"] in materias_ids
    ]

    return materias_completas

async def matricular(materia_id: int, tool_context: ToolContext):

    state = tool_context.state

    if "materias_matriculadas" not in state:
        state["materias_matriculadas"] = []

    if materia_id in state["materias_matriculadas"]:
        return {"erro": "Usuário já matriculado"}

    state["materias_matriculadas"].append(materia_id)

    return {"mensagem": "Matriculado com sucesso"}

async def desmatricular(materia_id: int, tool_context: ToolContext):

    state = tool_context.state

    if "materias_matriculadas" not in state:
        state["materias_matriculadas"] = []

    materias = state["materias_matriculadas"]

    if materia_id not in materias:
        return {"erro": "Usuário não está matriculado nessa matéria"}

    materias.remove(materia_id)

    return {
        "mensagem": "Desmatriculado com sucesso",
        "materias_restantes": materias
    }

Aqui são criados três agentes, o orquestrador, que vai receber o usuário e entender qual outro subagente chamar, o sumário, que vai ter a função de ver as informações das matérias disponíveis e o gerenciador de inscrição que vai ter as funções de ver as matérias matriculadas, se matricular em uma matéria e desmatricular em uma matéria.

In [5]:
agente_sumario = Agent(
    name="agente_sumario",
    model="gemini-2.5-flash",
    description="Acessa as matérias disponíveis e as informações de cada uma",
    instruction="Você um um sumário de matérias. "
                "Quando for feita uma requisição de todas as matérias disponíveis use a ferramenta 'get_materias' , "
                "Quando for feita uma requisição de uma matéria em especifica use a ferramenta 'get_materia', "
                "Faça apenas essa tarefa de recolher todas as matérias ou apenas uma. ",
    tools=[get_materias, get_materia],
)

agente_de_matricula = Agent(
    name="agente_de_matricula",
    model="gemini-2.5-flash",
    description="Auxilia o usuário nas tarefas de matrícula, desmatrícula e ver matérias matriculadas.",
    instruction="Você é um auxiliar de gerência de turmas matrículadas "
                "Quando o usuário pedir as matérias já matriculadas use a função 'get_materias_matriculadas', "
                "Quando o usuário pedir para se matricular em uma matéria use a função 'matricular' usando o ID de usuário e sessão já passados pelo sistema, "
                "Quando o usuário pedir para se desmatricular em uma matéria use a função 'desmatricular', "
                "Faça apenas essas tarefas.",
    tools=[get_materias_matriculadas, matricular, desmatricular], 
)
orquestrador = Agent(
    name="orquestrador",
    model="gemini-2.5-flash",
    description="Interpreta o que o usuário quer e quais agentes serão usados",
    instruction="Você é um assistente em um sistema de matrículas em matérias de uma faculdade, que recepciona o usuário e interpreta o que ele quer fazer"
                "Redirecione as tarefas de realizar a matrícula, retirar uma matrícula e olhar as matrículas de um estudante para o 'agente_de_matricula' "
                "Redirecione as tarefas de olhar as matérias disponíveis para o 'agente_sumario' "
                "Faça apenas a comunicação inicial e final com o usuário e redirecionamento de tarefas ",
                sub_agents=[agente_de_matricula, agente_sumario],
)

runner = Runner(
    agent=orquestrador,
    app_name="gerenciador_de_matricula",
    session_service=session_service
)

Função de chamada do agente principal.

In [6]:
async def chamar_orquestrador(query: str, user_id: str, session_id: str):
    content = types.Content(
        role="user",
        parts=[types.Part(text=query)]
    )

    resposta_final = "Nenhuma resposta final produzida."

    for event in runner.run(
        user_id=user_id,
        session_id=session_id,
        new_message=content
    ):

        if event.is_final_response():
            if event.content and event.content.parts:
                resposta_final = event.content.parts[0].text
            break

    return resposta_final

Chamada da função principal. É testado as funções principais diretamente na primeira e segunda query, na quarta e na quinta é testado se o modelo vai entender que ele precisa do ID e o que ele faz se receber ID e nome contraditórios, respectivamente.

In [7]:
query = "olá!!"
print(await chamar_orquestrador(query, "usuario_1", "sessao_001"))

query = "Quero olhar as matérias disponíveis"
print(await chamar_orquestrador(query, "usuario_1", "sessao_001"))

query = "Me dê os detalhes da matéria de Algoritmos"
print(await chamar_orquestrador(query, "usuario_1", "sessao_001"))

query = "Agora quero me matricular em uma matéria"
print(await chamar_orquestrador(query, "usuario_1", "sessao_001"))

query = "Agora quero me matricular em banco de dados com ID 1"
print(await chamar_orquestrador(query, "usuario_1", "sessao_001"))

query = "Perdão quero me matricular na matéria de ID 1"
print(await chamar_orquestrador(query, "usuario_1", "sessao_001"))

query = "quais turmas estou matriculado até então?"
print(await chamar_orquestrador(query, "usuario_1", "sessao_001"))

query = "Quero me desmatricular na matéria de algoritmos de ID 1."
print(await chamar_orquestrador(query, "usuario_1", "sessao_001"))

query = "Quais turmas ainda estou matriculado?"
print(await chamar_orquestrador(query, "usuario_1", "sessao_001"))

session2 = await session_service.create_session(
    app_name="gerenciador_de_matricula",
    user_id="usuario_2",
    session_id="sessao_002"
)

query = "quais turmas estou matriculado até então?"
print(await chamar_orquestrador(query, "usuario_2", "sessao_002"))




Olá! Bem-vindo ao sistema de matrículas. Como posso te ajudar hoje?


As matérias disponíveis são: Algoritmos (Prof. Dr. Carlos, 80 horas), Banco de Dados (Profa. Mariana, 60 horas) e Programação Orientada a Objetos (Prof. Dr. Renato, 80 horas).
A matéria de Algoritmos tem como professor o Dr. Carlos, possui carga horária de 80 horas e sua ementa é "Estruturas de dados básicas, análise de complexidade." A matéria está ativa.
Em qual matéria você gostaria de se matricular? Você gostaria de se matricular em Algoritmos?
O ID 1 corresponde à matéria de Algoritmos. Você gostaria de se matricular em Algoritmos ou em Banco de Dados, que tem o ID 2?
Você foi matriculado com sucesso na matéria de ID 1.
Você está matriculado em Algoritmos (ID: 1, Professor: Dr. Carlos, Carga Horária: 80 horas).
Você foi desmatriculado com sucesso da matéria de Algoritmos.
Parece que você ainda está matriculado em Algoritmos (ID: 1, Professor: Dr. Carlos, Carga Horária: 80 horas). Houve um erro na desmatrícula ou você se matriculou novamente?
Até o momento, você não está matriculad